In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
from pprint import pprint

from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

In [2]:
llm = OpenAI(model="gpt-4o", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))

# optional: keep Settings consistent for other llamaindex parts
Settings.llm = llm

In [3]:
from tavily import TavilyClient
from llama_index.core.tools import FunctionTool

def tavily_search(query: str):
    """Search the web using Tavily and return results."""
    client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    return client.search(query=query, search_depth="basic", max_results=5)

search_tool = FunctionTool.from_defaults(fn=tavily_search)

In [4]:
result = tavily_search("من هو محافظ مصرف ليبيا المركزي الحالي؟")
pprint(result)

{'answer': None,
 'follow_up_questions': None,
 'images': [],
 'query': 'من هو محافظ مصرف ليبيا المركزي الحالي؟',
 'request_id': 'ff157ae7-2be7-4268-aa42-bc9bc70970eb',
 'response_time': 0.94,
 'results': [{'content': 'محافظ مصرف ليبيا المركزي السيد ناجي محمد عيسى يجتمع '
                         'مع رئيس حكومة الوحدة الوطنية المهندس عبدالحميد '
                         'الدبيبة بمقر رئاسة الوزراء ظهر اليوم الأربعاء',
              'raw_content': None,
              'score': 0.99971753,
              'title': 'محافظ مصرف ليبيا المركزي السيد ناجي محمد عيسى يجتمع مع '
                       'رئيس ...',
              'url': 'https://www.facebook.com/CentralBankofLibya/posts/%D9%85%D8%AD%D8%A7%D9%81%D8%B8-%D9%85%D8%B5%D8%B1%D9%81-%D9%84%D9%8A%D8%A8%D9%8A%D8%A7-%D8%A7%D9%84%D9%85%D8%B1%D9%83%D8%B2%D9%8A-%D8%A7%D9%84%D8%B3%D9%8A%D8%AF-%D9%86%D8%A7%D8%AC%D9%8A-%D9%85%D8%AD%D9%85%D8%AF-%D8%B9%D9%8A%D8%B3%D9%89-%D9%8A%D8%AC%D8%AA%D9%85%D8%B9-%D9%85%D8%B9-%D8%B1%D8%A6%D9%8A%D8%B3-%D8%AD%D9%83%D9

In [5]:
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.workflow import Context

agent = ReActAgent(
    tools=[search_tool],
    llm=llm,
    verbose=True,
)

ctx = Context(agent)

In [6]:
handler = await agent.run("من هو محافظ مصرف ليبيا المركزي الحالي؟ أعطني الاسم مع رابط مصدر رسمي إن أمكن.", ctx=ctx)

print(handler.response)
print("\nTool calls:")
pprint(handler.tool_calls)

assistant: محافظ مصرف ليبيا المركزي الحالي هو ناجي محمد عيسى بلقاسم. يمكنك الاطلاع على المزيد من التفاصيل من خلال الرابط الرسمي التالي: [مجلس النواب الليبي يقر تعيين محافظ للمصرف المركزي](https://www.aljazeera.net/ebusiness/2024/10/1/%D9%85%D8%AC%D9%84%D8%B3-%D8%A7%D9%84%D9%86%D9%88%D8%A7%D8%A8-%D8%A7%D9%84%D9%84%D9%8A%D8%A8%D9%8A-%D9%8A%D9%82%D8%B1-%D8%AA%D8%B9%D9%8A%D9%8A%D9%86-%D9%85%D8%AD%D8%A7%D9%81%D8%B8%D8%A7).

Tool calls:
[ToolCallResult(tool_name='tavily_search', tool_kwargs={'query': 'محافظ مصرف ليبيا المركزي الحالي 2023'}, tool_id='87a8b80e-d4d5-4787-b7f2-4d5ef454b829', tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text="{'query': 'محافظ مصرف ليبيا المركزي الحالي 2023', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.facebook.com/bdallhsalm.204496/posts/%D9%81%D9%8A%D8%B3%D8%A8%D9%88%D9%83-%D9%8A%D8%B0%D9%83%D8%B1%D9%86%D9%8A-%D8%A8%D8%B0%D9%83%D8%B1%D9%8A%D8%A7%D8%AA-%D8%AC%D9%85%D9%8A%D9%84%D8%A9-%D8%AD%D8%AF%D8%AB